# Suite DOM — Value domains

A literal carries its value domain: a standard's representation (`Integer`, `Ieee754`, `Bits`, `Bytes`, `Unicode`,
`Ieee1164`, `Enum`, `Packed`, `Bool`), or by default its native's. Domains are data, compared by structure and
validated; a registered domain is written by name, any other by value, and a default one not at all. A value of another domain is a
typed value: compared within its domain, with integer and IEEE 754 arithmetic and the bitwise operations by domain.

In [ ]:
import { Domains as D, Evaluators as V, Expressions as E } from "@mbse/expressions";
import { Ieee754 } from "@mbse/expressions/Dialects/Basic";
import * as Python from "@mbse/expressions/Dialects/Python";
import { Domains as FD, Evaluators as FE, Terms as F } from "@mbse/expressions/Framework";
import { OverflowError } from "@mbse/expressions/Framework/Errors";
import * as Translators from "@mbse/expressions/Translators";
import { Comparison as C, Errors, JSON, Plain, Validators, YAML } from "@mbse/schemas/Framework";

const { LookupError, NotImplementedError, ValueError } = Errors;
import { assert, equal, map, raises, same } from "./support.js";

const I8 = new D.OfInteger.Builder().width(8n).create();
const U8 = new D.OfInteger.Builder().width(8n).signed(false).overflow("wrap").create();
const F32 = new D.OfIeee754.Builder().format("binary32").create();
const [F16, F128, D64, D128] = ["binary16", "binary128", "decimal64", "decimal128"].map((f) => new D.OfIeee754.Data(f));
const Bits2 = new D.OfBits.Builder().width(2n).create();
const State = new D.OfEnum.Builder().members("IDLE", "RUN").create();
const Coded = new D.OfPacked.Builder().domain(State).representation(Bits2).codes(0n, 1n).create();

/** A literal's snapshot, as plain data. */
function snapshot(literal: unknown): any {
  return (Plain.ToPlain(E.OfLiteral.Schema, literal as never) as any).get("objects").get("s0");
}

/** The value of a Basic operation, with the variables in `scope` bound. */
function evIn(scope: Record<string, unknown>, name: string, ...args: unknown[]): any {
  return V.OfAny(E.operation(name, ...args), scope);
}

/** The value of a Basic operation. */
function ev(name: string, ...args: unknown[]): any {
  return evIn({}, name, ...args);
}

/** Whether `value` is the typed value of `native` in `domain`. */
function is(value: unknown, domain: D.Domain, native: unknown): boolean {
  return new D.Value(domain, native).equals(value);
}

const bytes = (...values: number[]) => new Uint8Array(values);
const L = E.literal;
const x = E.variable("x");
const Nib = new D.OfBits.Data(4n);
const U = new D.OfInteger.Data(null, false);

## DOM-01 · Value domains name standards, with only their parameters, are compared by structure and validated

In [ ]:
{
  const names: [D.Domain, string][] = [[D.Bool, "bool"], [D.Int, "int"], [I8, "int8"], [U8, "uint8"],
    [new D.OfInteger.Builder().signed(false).create(), "uint"], [D.Float, "float"], [F32, "binary32"],
    [new D.OfIeee754.Builder().rounding("roundTowardZero").create(), "binary64 roundTowardZero"],
    [Bits2, "bits2"], [D.Bytes, "bytes"], [new D.OfBytes.Builder().width(4n).create(), "bytes4"], [D.Str, "str"],
    [new D.OfIeee1164.Builder().create(), "std_logic"], [State, "enum(IDLE, RUN)"], [Coded, "packed(enum(IDLE, RUN) as bits2)"]];
  assert(names.every(([domain, name]) => domain.name() === name && String(domain) === name));
  assert(names.every(([domain]) => domain.validate().length === 0));
  // The natives' defaults, compared by structure: an equal domain built anew is the same domain.
  assert(new D.OfInteger.Data().equals(D.Int) && new D.OfIeee754.Data("binary64").equals(D.Float) && new D.OfUnicode.Data().equals(D.Str));
  assert(new D.OfBytes.Data().equals(D.Bytes) && new D.OfBool.Data().equals(D.Bool) && !I8.equals(U8) && !I8.equals(D.Int));
  assert(new D.OfInteger.Builder().width(8n).create().equals(I8) && String(D.OfInteger) === "<domain kind integer>");
  assert([D.OfBool, D.OfInteger, D.OfIeee754, D.OfBits, D.OfBytes, D.OfUnicode, D.OfIeee1164, D.OfEnum, D.OfPacked]
    .every((kind) => kind.Schema.validate().length === 0) && D.Schema.validate().length === 0);
  assert(same(D.Schema.branches.map((b) => b.name), ["bool", "integer", "ieee754", "bits", "bytes", "unicode", "ieee1164",
    "enum", "packed", "named"]));
  assert(D.FORMATS[2] === "binary64" && D.ROUNDINGS[0] === "roundTiesToEven" && same(D.OVERFLOWS, ["wrap", "saturate", "raise"]));
  assert(D.STATES.length === 9);
  // Each kind's problems.
  const problems: [D.Domain, string[]][] = [
    [new D.OfInteger.Data(0n), ["a width must be a positive int, got 0"]],
    [new D.OfInteger.Data(null, false, "wrap"), ["an integer without a width cannot wrap"]],
    [new D.OfInteger.Data(8n, "yes" as never, "clip"), ["signed must be a bool, got 'yes'",
      "overflow must be one of wrap, saturate, raise, got 'clip'"]],
    [new D.OfIeee754.Data("binary8", "up"), [
      "format must be one of binary16, binary32, binary64, binary128, decimal64, decimal128, got 'binary8'",
      "rounding must be one of roundTiesToEven, roundTiesToAway, roundTowardPositive, roundTowardNegative, " +
      "roundTowardZero, got 'up'"]],
    [new D.OfBits.Data(), ["a bits domain needs a width"]],
    [new D.OfBits.Data(0n), ["a width must be a positive int, got 0"]],
    [new D.OfBytes.Data(-1n), ["a width must be a positive int, got -1"]],
    [new D.OfEnum.Data(), ["an enum needs a member"]],
    [new D.OfEnum.Data(["A", "", "A"]), ["member 1 has no name", "member 'A' appears twice"]],
    [new D.OfPacked.Data(I8, Bits2, []), ["a packed domain packs an enum, got int8"]],
    [new D.OfPacked.Data(), ["a packed domain packs an enum, got None"]],
    [new D.OfPacked.Data(State, D.Bytes, [0n, 1n]), ["a packed domain's representation is an integer or bits of a width, got bytes"]],
    [new D.OfPacked.Data(State, D.Int, [0n, 1n]), ["a packed domain's representation is an integer or bits of a width, got int"]],
    [new D.OfPacked.Data(State, Bits2, [0n]), ["a packed domain needs a code per member: 2 members, 1 codes"]],
    [new D.OfPacked.Data(State, new D.OfBits.Data(1n), [0n, 2n]), ["code 1 does not fit bits1: 2"]],
    [new D.OfPacked.Data(State, Bits2, [1n, 1n]), ["code 1 is used twice"]],
    [new D.OfPacked.Data(State, new D.OfInteger.Data(2n), [-2n, 2n]), ["code 1 does not fit int2: 2"]],
    [new D.OfPacked.Data(State, new D.OfInteger.Data(2n), ["0" as never, 1n]), ["code 0 does not fit int2: '0'"]],
    [new D.OfPacked.Data(new D.OfEnum.Data(), new D.OfInteger.Data(2n, true, "x"), []), [
      "domain: an enum needs a member", "representation: overflow must be one of wrap, saturate, raise, got 'x'"]],
  ];
  for (const [domain, expected] of problems) assert(same(domain.validate(), expected), `${domain}: ${domain.validate()}`);
  assert(new D.OfPacked.Data().name() === "packed(None as None)");
  // Builders finalize as every builder does, and set each parameter.
  const built = new D.OfIeee754.Builder(F32).rounding("roundTowardNegative").clone();
  assert(built.equals(new D.OfIeee754.Data("binary32", "roundTowardNegative")) && F32.rounding === "roundTiesToEven");
  assert(new D.OfBytes.Builder(new D.OfBytes.Data()).width(3n).update().width === 3n);
  for (const builder of [new D.OfBool.Builder(), new D.OfUnicode.Builder(), new D.OfIeee1164.Builder()]) {
    assert(builder.create().validate().length === 0);
  }
  raises(ValueError, () => new D.OfInteger.Builder(I8).create(), "create() is only valid without a source instance; use clone() or update()");
  raises(ValueError, () => new D.OfInteger.Builder().clone(), "clone() is only valid with a source instance");
  raises(ValueError, () => new D.OfInteger.Builder().update(), "update() is only valid with a source instance");
  raises(TypeError, () => new D.OfInteger.Builder(Bits2 as never), "expected integer data to build from, got bits2");
}

## DOM-02 · A domain holds the values of its native type that fit it, and includes only itself

In [ ]:
{
  const bytes = (...values: number[]) => new Uint8Array(values);
  const holds: [D.Domain, unknown[], unknown[]][] = [
    [I8, [127n, -128n], [128n, -129n, true, 1.0]], [U8, [0n, 255n], [-1n, 256n]], [D.Int, [10n ** 30n, -5n], [false]],
    [new D.OfInteger.Data(null, false), [10n ** 30n], [-1n]], [new D.OfInteger.Data(0n), [10n ** 30n], []], // not a width: unbounded
    [F32, [1.5, NaN], [1n, "1.5"]], [new D.OfIeee754.Data("decimal64"), ["0.1"], [0.1]],
    [new D.OfBits.Data(3n), [bytes(7), bytes(0)], [bytes(8), bytes(0, 7), "7"]], [new D.OfBits.Data(8n), [bytes(255)], [bytes()]],
    [new D.OfBits.Data(9n), [bytes(1, 255)], [bytes(2, 0), bytes(255)]], [new D.OfBits.Data(), [], [bytes(0)]],
    [new D.OfBytes.Data(2n), [bytes(97, 98)], [bytes(97)]], [D.Bytes, [bytes(), bytes(97, 98, 99)], ["abc"]],
    [D.Str, ["", "日本語 🚀"], ["\ud800", bytes(97)]], [new D.OfIeee1164.Data(), ["Z", "-", "U"], ["z", "2", 1n]],
    [State, ["RUN"], ["STOP", 0n]], [Coded, ["IDLE"], ["STOP"]], [new D.OfPacked.Data(), [], ["IDLE"]],
    [D.Bool, [true, false], [1n, 0n]],
  ];
  for (const [domain, inside, outside] of holds) {
    assert(inside.every((v) => domain.contains(v)) && !outside.some((v) => domain.contains(v)), String(domain));
  }
  assert(new D.OfIeee754.Data("decimal128").native() === String && F32.native() === Number && I8.native() === BigInt);
  // A domain includes only domains equal to it: Basic has no implied promotions.
  assert(I8.includes(new D.OfInteger.Data(8n)) && !I8.includes(D.Int) && !D.Int.includes(I8) && !I8.includes(U8));
  assert(I8.includes(new FD.OfUnion(I8, new D.OfInteger.Data(8n))) && !I8.includes(new FD.OfUnion(I8, U8)));
  assert(FD.overlaps(FD.Anything, I8) && !D.Float.includes(F32));
  // A native's default domain, and the domains of other values.
  assert(same([true, 1n, 1.5, "a", bytes(97)].map((v) => D.of(v)), [D.Bool, D.Int, D.Float, D.Str, D.Bytes]));
  assert(D.of(E.variable("x").data) === D.Object);
  raises(TypeError, () => D.of(null), "a NoneType is not a value of the Basic dialect");
}

## DOM-03 · Domains are registered by name, once, and written by name; any other by value

In [ ]:
{
  D.register("uint8", U8);
  assert(D.registered("uint8") === U8 && D.name_of(U8) === "uint8" && D.name_of(new D.OfInteger.Data(8n, false, "wrap")) === "uint8");
  assert(D.name_of(I8) === null);
  raises(ValueError, () => D.register("uint8", I8), "domain 'uint8' is already registered");
  raises(ValueError, () => D.register("byte", new D.OfInteger.Data(8n, false, "wrap")), "uint8 is already registered as 'uint8'");
  D.register("uint16", new D.OfInteger.Data(16n, false)); // another domain: each name, and each domain, once
  assert(D.name_of(new D.OfInteger.Data(16n, false)) === "uint16");
  raises(LookupError, () => D.registered("nope"), "no domain registered as 'nope'");
  assert(equal(D.to_plain(U8), map({ named: { name: "uint8" } })));
  assert(equal(D.to_plain(I8), map({ integer: { width: 8n, signed: true, overflow: "raise" } })));
  assert(equal(D.to_plain(D.Int), map({ integer: { signed: true, overflow: "raise" } })));
  assert(equal(D.to_plain(new D.OfPacked.Data(State, U8, [0n, 1n])), map({
    packed: { domain: { enum: { members: ["IDLE", "RUN"] } }, representation: { named: { name: "uint8" } }, codes: [0n, 1n] } })));
  assert(equal(D.to_plain(new D.OfUnicode.Data()), map({ unicode: {} })));
  for (const domain of [U8, I8, F32, Bits2, D.Bytes, D.Str, State, Coded, new D.OfPacked.Data(State, U8, [0n, 1n]), D.Bool]) {
    assert(D.from_plain(D.to_plain(domain)).equals(domain), String(domain));
  }
  raises(TypeError, () => D.to_plain("int8"), "not a domain: 'int8'");
  raises(LookupError, () => D.from_plain(map({ named: { name: "nope" } })), "no domain registered as 'nope'");
}

## DOM-04 · A literal carries its domain; a native's default domain is no domain at all

In [ ]:
{
  const lit = E.literal(200n, U8).data as E.OfLiteral.Data;
  assert(lit.domain === U8 && lit.typed() === U8 && lit.validate().length === 0);
  assert((E.literal(5n, D.Int).data as E.OfLiteral.Data).domain === null && E.literal(5n).data.typed() === null); // the default is left out
  assert((new E.OfLiteral.Builder().value(-1n).domain(I8).create().domain as D.Domain).equals(I8));
  assert(new E.OfLiteral.Builder(lit).value(7n).clone().domain === U8 && new E.OfLiteral.Builder(lit).domain(null).clone().domain === null);
  assert(new E.OfLiteral.Builder(E.literal(1n).data as E.OfLiteral.Data).domain(D.Int).update().domain === null);
  // A literal's domain is valid and holds its value.
  assert(same(E.literal(300n, U8).data.validate(), ["a literal of uint8 cannot hold 300"]));
  assert(same(E.literal(new Uint8Array([0]), new D.OfBits.Data()).data.validate(), ["domain: a bits domain needs a width"]));
  assert(same(new E.OfLiteral.Data(1n, "int8").validate(), ["a literal's domain must be a value domain, got str"]));
  assert(same(E.operation("eq", E.literal("RUN", Coded), E.literal("STOP", Coded)).data.validate(),
    ["argument 1: a literal of packed(enum(IDLE, RUN) as bits2) cannot hold 'STOP'"]));
  // Inference gives a literal's domain, and Basic does not promote: int8 and int8 add, int8 and int16 do not.
  assert(E.DIALECT.infer(lit) === U8 && E.DIALECT.infer(E.literal(1n)) === D.Int);
  assert((E.DIALECT.infer(E.operation("add", E.literal(1n, I8), E.literal(2n, I8))) as D.Domain).equals(I8));
  raises(TypeError, () => E.DIALECT.infer(E.operation("add", E.literal(1n, I8), E.literal(2n, new D.OfInteger.Data(16n)))),
    "add cannot take (int8, int16); it takes (T, T) -> T for T in int | float or any integer or ieee754 domain");
  // Forms, `make` and `same` carry the domain.
  const attributes = lit.form().attributes;
  assert(attributes.size === 2 && attributes.get("value") === 200n && attributes.get("domain") === U8);
  assert(F.same(E.DIALECT.make(lit.form()), lit) && F.same(lit, E.literal(200n, new D.OfInteger.Data(8n, false, "wrap")).data));
  assert(!F.same(lit, E.literal(200n, I8).data) && !F.same(lit, E.literal(200n).data));
  // A literal of another domain evaluates to a typed value; an evaluator without typed values refuses it.
  assert(new D.Value(U8, 200n).equals(V.OfAny(lit)));
  raises(NotImplementedError, () => new FE.Interpreter(E.DIALECT, new Map([["operation", V.OPERATIONS]])).run(lit),
    "this evaluator has no values of uint8");
  assert(V.OfAny(E.operation("add", E.literal(1n, D.Int), 2n)) === 3n);
  // Translation keeps a domain only where the other dialect has one: none does yet.
  raises(ValueError, () => Translators.between(E.DIALECT, Python.Expressions.DIALECT).forward(lit),
    "Basic literal 200 of uint8 has no Python counterpart");
}

## DOM-05 · Over the wire a literal's domain is written by name when registered, otherwise by value

In [ ]:
{
  assert(equal(snapshot(E.literal(200n, U8).data), map({ kind: "literal", int: 200n, domain: { named: { name: "uint8" } } })));
  assert(equal(snapshot(E.literal(-1n, I8).data), map({
    kind: "literal", int: -1n, domain: { integer: { width: 8n, signed: true, overflow: "raise" } } })));
  assert(equal(snapshot(E.literal(5n, D.Int).data), map({ kind: "literal", int: 5n })));
  const text = JSON.ToJSON(E.OfLiteral.Schema, E.literal("RUN", Coded).data as never);
  assert(text === '{"root": "s0", "objects": {"s0": {"kind": "literal", "str": "RUN", "domain": {"packed": {"domain": ' +
    '{"enum": {"members": ["IDLE", "RUN"]}}, "representation": {"bits": {"width": 2}}, "codes": [0, 1]}}}}}');
  for (const literal of [E.literal(200n, U8).data, E.literal(-1n, I8).data, E.literal("RUN", Coded).data, E.literal(1.5, F32).data,
    E.literal("Z", new D.OfIeee1164.Data()).data, E.literal(new Uint8Array([3]), Bits2).data] as E.OfLiteral.Data[]) {
    const back = JSON.FromJSON(E.Builders as never)(E.OfLiteral.Schema, JSON.ToJSON(E.OfLiteral.Schema, literal as never)) as E.OfLiteral.Data;
    assert(F.same(back, literal) && (back.domain as D.Domain).equals(literal.domain));
    assert(F.same(YAML.FromYAML(E.Builders as never)(E.OfLiteral.Schema, YAML.ToYAML(E.OfLiteral.Schema, literal as never)), literal));
    assert(Validators.Validate(E.Builders as never)(E.OfLiteral.Schema, back).length === 0);
  }
  assert((JSON.FromJSON(E.Builders as never)(E.OfLiteral.Schema, JSON.ToJSON(E.OfLiteral.Schema, E.literal(1n, U8).data as never)) as
    E.OfLiteral.Data).domain === U8);
  raises(LookupError, () => Plain.FromPlain(E.Builders as never)(E.OfLiteral.Schema, map({ root: "s0", objects: { s0: {
    kind: "literal", int: 1n, domain: { named: { name: "nope" } } } } })), "no domain registered as 'nope'");
  // Literals compare under their meta-schema with their domains.
  const compare = (a: unknown, b: unknown) => new C.OfObject(E.OfLiteral.Schema, a as never).compare(new C.OfObject(E.OfLiteral.Schema, b as never));
  assert(compare(E.literal(1n, I8).data, E.literal(1n, new D.OfInteger.Data(8n)).data) === 0);
  assert(compare(E.literal(1n, I8).data, E.literal(1n, U8).data) === null && compare(E.literal(1n, I8).data, E.literal(1n).data) === null);
  // The builder reads and writes the domain through the visitor protocols too.
  const domainField = (builder: any, branch: string, name: string, act: (n: any) => unknown) =>
    builder.property("domain", (p: any) => p.value((a: any) => a.as_union((u: any) => u.property(branch, (q: any) => q.value(
      (b: any) => b.as_object((o: any) => o.property(name, (w: any) => w.value((n: any) => n.as_native(act)))))))));
  const builder = new E.OfLiteral.Builder(E.literal(-1n, I8).data as E.OfLiteral.Data);
  const names: string[] = [];
  builder.properties((p) => names.push(p.name()));
  assert(builder.has("domain") && same(names, ["kind", "int", "domain"]));
  const widths: unknown[] = [];
  domainField(builder, "integer", "width", (n) => widths.push(n.get()));
  assert(same(widths, [8n]));
  domainField(builder, "integer", "width", (n) => n.set(16n));
  assert((builder.clone().domain as D.Domain).equals(new D.OfInteger.Data(16n)));
  builder.property("domain", (p) => p.clear());
  assert(!builder.has("domain") && builder.clone().domain === null);
  domainField(builder, "named", "name", (n) => n.set("uint8"));
  assert(builder.clone().domain === U8);
  builder.clear("domain");
  assert(builder.clone().domain === null);
  builder.property("domain", (p) => p.value((a) => a.as_union(() => null))); // a union without a branch is no value
  assert(builder.clone().domain === null);
}

## DOM-06 · A value of another domain is a typed value, compared only within its domain

In [ ]:
{
  assert(new D.Value(U8, 200n).equals(D.value(U8, 200n)) && D.value(D.Int, 5n) === 5n && typeof D.value(D.Bool, true) === "boolean");
  assert(D.of(new D.Value(U8, 1n)) === U8 && String(new D.Value(U8, 200n)) === "200 as uint8");
  raises(ValueError, () => new D.Value(U8, 256n), "uint8 cannot hold 256");
  raises(TypeError, () => new D.Value("uint8" as never, 1n), "not a value domain: 'uint8'");
  // eq and ne within one domain; other domains, typed or default, are incomparable.
  assert(ev("eq", L(5n, I8), L(5n, I8)) === true && ev("ne", L(5n, I8), L(6n, I8)) === true);
  assert(ev("eq", L(5n, I8), 5n) === null && ev("eq", L(5n, I8), L(5n, U8)) === null && ev("lt", L(5n, I8), 6n) === null);
  assert(ev("lt", L(5n, I8), L(5n, U8)) === null);
  // Ordered: integers, IEEE 754 numbers, bytes and packed enums (by code); unordered: bits, enums and std_logic.
  assert(ev("lt", L(-1n, I8), L(1n, I8)) === true && ev("ge", L(1.5, F32), L(2.5, F32)) === false);
  const Bytes2 = new D.OfBytes.Data(2n);
  assert(ev("lt", L(bytes(0, 1), Bytes2), L(bytes(1, 0), Bytes2)) === true);
  const Reversed = new D.OfPacked.Data(State, Bits2, [1n, 0n]);
  assert(ev("gt", L("IDLE", Reversed), L("RUN", Reversed)) === true && ev("lt", L("IDLE", Coded), L("RUN", Coded)) === true);
  assert(ev("eq", L("RUN", Coded), L("RUN", Coded)) === true);
  assert(ev("lt", L(bytes(1), Bits2), L(bytes(2), Bits2)) === null && ev("eq", L(bytes(1), Bits2), L(bytes(1), Bits2)) === true);
  assert(ev("lt", L("IDLE", State), L("RUN", State)) === null && ev("ne", L("IDLE", State), L("RUN", State)) === true);
  const Logic = new D.OfIeee1164.Data();
  assert(ev("eq", L("X", Logic), L("X", Logic)) === true && ev("le", L("0", Logic), L("1", Logic)) === null);
  // A format's floats compare as mbse-schemas compares floats: NaNs equal each other, and -0 comes before 0.
  assert(ev("eq", L(NaN, F32), L(NaN, F32)) === true && ev("lt", L(NaN, F32), L(1.0, F32)) === null);
  assert(ev("lt", L(-0, F32), L(0, F32)) === true);
  assert(ev("eq", L("1.5", new D.OfIeee754.Data("decimal64")), L("1.5", new D.OfIeee754.Data("decimal64"))) === true); // see DOM-12
  // A scope may bind a variable to a typed value; unknown stays unknown.
  assert(V.OfAny(E.variable("x").eq(L(3n, I8)), { x: new D.Value(I8, 3n) }) === true);
  assert(V.OfAny(E.variable("x").lt(L(3n, I8)), { x: null }) === null);
}

## DOM-07 · Integer arithmetic is exact, then the domain's overflow wraps, saturates or raises

In [ ]:
{
  assert(is(ev("add", L(100n, I8), L(27n, I8)), I8, 127n));
  raises(OverflowError, () => ev("add", L(100n, I8), L(28n, I8)), "add overflows int8: 128");
  raises(OverflowError, () => ev("neg", L(-128n, I8)), "neg overflows int8: 128");
  assert(is(ev("add", L(250n, U8), L(10n, U8)), U8, 4n) && is(ev("sub", L(0n, U8), L(1n, U8)), U8, 255n));
  const W8 = new D.OfInteger.Data(8n, true, "wrap");
  assert(is(ev("mul", L(64n, W8), L(2n, W8)), W8, -128n) && is(ev("neg", L(-128n, W8)), W8, -128n));
  const S8 = new D.OfInteger.Data(8n, true, "saturate");
  assert(is(ev("add", L(100n, S8), L(100n, S8)), S8, 127n) && is(ev("sub", L(-100n, S8), L(100n, S8)), S8, -128n));
  // An unbounded unsigned integer has a least value only, and without a width wrap does not apply.
  assert(is(ev("add", L(10n ** 30n, U), L(1n, U)), U, 10n ** 30n + 1n));
  const US = new D.OfInteger.Data(null, false, "saturate");
  assert(is(ev("sub", L(1n, US), L(2n, US)), US, 0n));
  raises(OverflowError, () => ev("sub", L(1n, U), L(2n, U)), "sub overflows uint: -1");
  const UW = new D.OfInteger.Data(null, false, "wrap"); // invalid, as validation reports; evaluation raises
  raises(OverflowError, () => ev("sub", L(1n, UW), L(2n, UW)), "sub overflows uint: -1");
  // Operands of different domains raise: Basic does not promote.
  raises(TypeError, () => ev("add", L(1n, I8), 1n), "add expects numbers of one domain, got int8 and int");
  raises(TypeError, () => ev("add", L(1n, I8), L(1n, U8)), "add expects numbers of one domain, got int8 and uint8");
  raises(TypeError, () => ev("neg", L("IDLE", State)), "neg expects a number, got enum(IDLE, RUN)");
  // The default domains' arithmetic is unchanged; IEEE 754 formats round in their own (see DOM-11).
  assert(ev("add", 2n ** 70n, 1n) === 2n ** 70n + 1n && ev("mul", 1.5, 2.0) === 3.0);
  assert(is(ev("add", L(1.5, F32), L(1.5, F32)), F32, 3.0));
}

## DOM-08 · The bitwise operations take integers or bits of one domain, and shifts a count of any integer domain

In [ ]:
{
  assert(ev("bitand", 12n, 10n) === 8n && ev("bitor", 12n, 10n) === 14n && ev("bitxor", 12n, 10n) === 6n && ev("bitnot", 5n) === -6n);
  assert(is(ev("bitnot", L(5n, U8)), U8, 250n) && is(ev("bitnot", L(5n, I8)), I8, -6n));
  assert(is(ev("bitand", L(-1n, I8), L(15n, I8)), I8, 15n));
  assert(is(ev("bitxor", L(bytes(0x0c), Nib), L(bytes(0x0a), Nib)), Nib, bytes(0x06)));
  assert(is(ev("bitor", L(bytes(0x0c), Nib), L(bytes(0x03), Nib)), Nib, bytes(0x0f)));
  assert(is(ev("bitand", L(bytes(0x0c), Nib), L(bytes(0x0a), Nib)), Nib, bytes(0x08)));
  assert(is(ev("bitnot", L(bytes(0x05), Nib)), Nib, bytes(0x0a))); // within the width
  // Without a width, a result outside the domain applies its overflow.
  raises(OverflowError, () => ev("bitnot", L(5n, U)), "bitnot overflows uint: -6");
  raises(TypeError, () => ev("bitand", L(1n, I8), L(bytes(1), Nib)), "bitand expects integers or bits of one domain, got int8 and bits4");
  raises(TypeError, () => ev("bitnot", true), "bitnot expects an integer or bits, got bool");
  // Shifts: on integers by a power of two, with the overflow (shr toward negative infinity); on bits, logical.
  assert(ev("shl", 3n, 4n) === 48n && ev("shr", -7n, 1n) === -4n && is(ev("shr", L(-128n, I8), L(7n, U8)), I8, -1n));
  assert(is(ev("shl", L(1n, U8), 7n), U8, 128n) && is(ev("shl", L(1n, U8), 8n), U8, 0n));
  raises(OverflowError, () => ev("shl", L(1n, I8), 7n), "shl overflows int8: 128");
  assert(is(ev("shl", L(bytes(0x09), Nib), 1n), Nib, bytes(0x02)) && is(ev("shr", L(bytes(0x09), Nib), 3n), Nib, bytes(0x01)));
  raises(ValueError, () => ev("shl", 1n, -1n), "shl needs a non-negative count, got -1");
  raises(TypeError, () => ev("shr", 1n, 1.0), "shr expects an integer count, got float");
  raises(TypeError, () => ev("shl", "a", 1n), "shl expects an integer or bits, got str");
  assert(evIn({ x: null }, "bitand", x, 1n) === null && evIn({ x: null }, "shl", x, 1n) === null);
  assert(evIn({ x: null }, "shl", 1n, x) === null);
}

## DOM-09 · Signatures take any domain of a family, so inference follows typed values

In [ ]:
{
  const infer = (expression: unknown, environment?: Record<string, FD.Domain>) => E.DIALECT.infer(expression, environment);
  assert(infer(E.operation("eq", L("IDLE", State), L("RUN", State))) === D.Bool);
  assert(infer(E.operation("lt", L(1n, I8), L(2n, I8))) === D.Bool);
  raises(TypeError, () => infer(E.operation("lt", L("IDLE", State), L("RUN", State))),
    "lt cannot take (enum(IDLE, RUN), enum(IDLE, RUN)); it takes (T, T) -> bool for T in " +
    "int | float | str | bytes or any integer, ieee754, bytes, unicode or packed domain");
  assert((infer(E.operation("mul", L(1.5, F32), x), { x: FD.Anything }) as D.Domain).equals(F32)); // an unknown takes the other's domain
  assert((infer(E.operation("bitxor", L(bytes(1), Nib), L(bytes(2), Nib))) as D.Domain).equals(Nib) && infer(E.operation("bitnot", 1n)) === D.Int);
  assert((infer(E.operation("shl", L(1n, U8), L(2n, I8))) as D.Domain).equals(U8) && infer(E.operation("shr", x, 1n), { x: FD.Anything }) === D.Int);
  assert(infer(E.operation("shl", 1n, x), { x: new FD.OfUnion(D.Int, I8) }) === D.Int);
  raises(TypeError, () => infer(E.operation("shl", 1n, 1.5)),
    "shl cannot take (int, float); it takes (T, any integer domain) -> T for T in int or any integer or bits domain");
  raises(TypeError, () => infer(E.operation("shr", 1.5, 1n)), "shr cannot take (float, int)");
  assert(E.operation("bitand", 1n, 2n).data.validate({ core: true }).length === 0);
  // A family includes every domain of its kinds, and holds their values.
  const family = (D.SIGNATURES.get("add") as FD.Same).family as FD.Domain;
  assert(String(family) === "integer or ieee754 domain" && family.contains(new D.Value(I8, 1n)) && family.contains(1.5));
  assert(!family.contains("a") && !family.contains(new D.Value(State, "RUN")));
}

## DOM-10 · An IEEE 754 value is a float up to binary64 and canonical text otherwise, which the format must hold

In [ ]:
{
  const holds: [D.Domain, unknown[], unknown[]][] = [
    [F16, [1.5, 65504.0, 2 ** -24, -0, Infinity, NaN], [0.1, 1e10, 2 ** -25, "1.5"]],
    [F32, [0.10000000149011612], [0.1]],
    [new D.OfIeee754.Data("binary64", "roundTowardZero"), [0.1, 5e-324], ["0.1"]],
    [F128, ["0.1", "1E+2", "-0", "Infinity", "-Infinity", "NaN", "1.189731495357231765085759326628007E+4932"],
      ["0.10", "100", "1e2", "-NaN", "1E+5000", "0.1 ", 0.1]],
    [D64, ["1.50", "1.000000000000000E+16", "0E-398", "-0", "0.000001", "1E-7", "NaN"],
      ["1.5E+0", "12345678901234567", "1E+385", "1E-399", "0.0000001", "+1", "abc"]],
    [D128, ["1.000000000000000000000000000000000", "2E+6111"], ["1E+6112"]],
  ];
  for (const [domain, inside, outside] of holds) {
    assert(inside.every((v) => domain.contains(v)) && !outside.some((v) => domain.contains(v)), String(domain));
  }
  assert(F128.native() === String && D64.native() === String && F16.native() === Number);
  assert(!new D.OfIeee754.Data("binary8").contains(1.5)); // an unknown format holds nothing
  // A decimal value's text is General Decimal Arithmetic's scientific form, keeping the coefficient and the exponent.
  const data: [boolean, bigint, number][] = [[false, 15n, -1], [true, 150n, -2], [false, 1n, 2], [false, 123n, -9], [false, 0n, 3], [true, 0n, 0]];
  assert(same(data.map(([n, c, q]) => Ieee754.text(new Ieee754.Datum(null, n, c, q))), ["1.5", "-1.50", "1E+2", "1.23E-7", "0E+3", "-0"]));
  assert(Ieee754.text(Ieee754.NAN) === "NaN" && same([...Ieee754.FORMATS.get("decimal64")!], [10n, 16, 384]));
}

## DOM-11 · IEEE 754 arithmetic is exact, then rounded to the format by the domain's rounding direction

In [ ]:
{
  const [EVEN, AWAY, UP, DOWN, ZERO] = D.ROUNDINGS;
  const TABLE: [string, string, string, unknown, unknown, unknown][] = [ // format, rounding, operation, operands, result
    ["binary16", EVEN, "add", 1.0, 0.00048828125, 1.0],
    ["binary16", EVEN, "add", 1.0009765625, 0.00048828125, 1.001953125],
    ["binary16", AWAY, "add", 1.0, 0.00048828125, 1.0009765625],
    ["binary16", UP, "add", 1.0, 5.960464477539063e-08, 1.0009765625],
    ["binary16", DOWN, "add", 1.0, 5.960464477539063e-08, 1.0],
    ["binary16", ZERO, "add", 1.0, 5.960464477539063e-08, 1.0],
    ["binary16", DOWN, "add", -1.0, -5.960464477539063e-08, -1.0009765625],
    ["binary16", UP, "add", -1.0, -5.960464477539063e-08, -1.0],
    ["binary16", EVEN, "add", 65504.0, 65504.0, Infinity],
    ["binary16", ZERO, "add", 65504.0, 65504.0, 65504.0],
    ["binary16", UP, "add", 65504.0, 65504.0, Infinity],
    ["binary16", DOWN, "add", 65504.0, 65504.0, 65504.0],
    ["binary16", DOWN, "add", -65504.0, -65504.0, -Infinity],
    ["binary16", UP, "add", -65504.0, -65504.0, -65504.0],
    ["binary16", AWAY, "mul", 65504.0, 2.0, Infinity],
    ["binary16", EVEN, "mul", 6.103515625e-05, 0.0009765625, 5.960464477539063e-08],
    ["binary16", EVEN, "mul", 5.960464477539063e-08, 0.5, 0.0],
    ["binary16", AWAY, "mul", 5.960464477539063e-08, 0.5, 5.960464477539063e-08],
    ["binary16", EVEN, "add", 2047.0, 0.5, 2048.0],
    ["binary16", EVEN, "sub", 1.0, 1.0, 0.0],
    ["binary16", DOWN, "sub", 1.0, 1.0, -0],
    ["binary16", EVEN, "add", -0, -0, -0],
    ["binary16", EVEN, "add", 0.0, -0, 0.0],
    ["binary16", EVEN, "mul", -0, 5.0, -0],
    ["binary16", EVEN, "sub", Infinity, Infinity, NaN],
    ["binary16", EVEN, "add", Infinity, 1.0, Infinity],
    ["binary16", EVEN, "add", 1.0, -Infinity, -Infinity],
    ["binary16", EVEN, "add", NaN, 1.0, NaN],
    ["binary16", EVEN, "mul", Infinity, 0.0, NaN],
    ["binary16", EVEN, "mul", 0.0, -Infinity, NaN],
    ["binary16", EVEN, "mul", Infinity, -2.0, -Infinity],
    ["binary16", EVEN, "mul", NaN, 1.0, NaN],
    ["binary16", EVEN, "neg", 1.5, null, -1.5],
    ["binary16", EVEN, "neg", NaN, null, NaN],
    ["binary16", EVEN, "neg", -0, null, 0.0],
    ["binary32", EVEN, "add", 16777216.0, 1.0, 16777216.0],
    ["binary32", UP, "add", 16777216.0, 1.0, 16777218.0],
    ["binary32", EVEN, "mul", 0.10000000149011612, 3.0, 0.30000001192092896],
    ["binary64", ZERO, "add", 0.1, 0.2, 0.3],
    ["binary64", UP, "add", 0.1, 0.2, 0.30000000000000004],
    ["binary64", DOWN, "mul", 0.3333333333333333, 3.0, 0.9999999999999999],
    ["binary128", EVEN, "add", "0.1", "0.2", "0.30000000000000000000000000000000004"],
    ["binary128", EVEN, "mul", "1E+4932", "1E+1", "Infinity"],
    ["binary128", ZERO, "mul", "1E+4932", "1E+1", "1.189731495357231765085759326628007E+4932"],
    ["binary128", EVEN, "sub", "1", "1", "0"],
    ["binary128", DOWN, "sub", "1", "1", "-0"],
    ["binary128", EVEN, "neg", "-0", null, "0"],
    ["binary128", EVEN, "add", "Infinity", "1", "Infinity"],
    ["binary128", EVEN, "mul", "1E-4960", "1E-10", "0"],
    ["decimal64", EVEN, "add", "1.5", "1.50", "3.00"],
    ["decimal64", EVEN, "mul", "1.5", "1.50", "2.250"],
    ["decimal64", EVEN, "add", "9999999999999999", "1", "1.000000000000000E+16"],
    ["decimal64", EVEN, "add", "9999999999999999", "0.5", "1.000000000000000E+16"],
    ["decimal64", AWAY, "add", "1000000000000000", "0.5", "1000000000000001"],
    ["decimal64", EVEN, "add", "1000000000000000", "0.5", "1000000000000000"],
    ["decimal64", UP, "add", "1000000000000000", "0.1", "1000000000000001"],
    ["decimal64", DOWN, "add", "-1000000000000000", "-0.1", "-1000000000000001"],
    ["decimal64", ZERO, "add", "-1000000000000000", "-0.9", "-1000000000000000"],
    ["decimal64", EVEN, "mul", "1E+369", "1E+10", "1.0000000000E+379"],
    ["decimal64", EVEN, "add", "9.999999999999999E+384", "1E+369", "Infinity"],
    ["decimal64", ZERO, "add", "9.999999999999999E+384", "1E+369", "9.999999999999999E+384"],
    ["decimal64", EVEN, "mul", "1E-398", "0.1", "0E-398"],
    ["decimal64", UP, "mul", "1E-398", "0.1", "1E-398"],
    ["decimal64", EVEN, "mul", "1E-390", "0.00001", "1E-395"],
    ["decimal64", EVEN, "sub", "1.5", "1.5", "0.0"],
    ["decimal64", DOWN, "sub", "1.5", "1.5", "-0.0"],
    ["decimal64", EVEN, "add", "-0", "-0", "-0"],
    ["decimal64", EVEN, "mul", "-0.0", "1E+5", "-0E+4"],
    ["decimal64", EVEN, "mul", "1E+369", "0E+10", "0E+369"],
    ["decimal64", EVEN, "add", "NaN", "1", "NaN"],
    ["decimal64", EVEN, "mul", "Infinity", "0", "NaN"],
    ["decimal64", EVEN, "neg", "1.50", null, "-1.50"],
    ["decimal64", EVEN, "add", "0.000001", "1E-7", "0.0000011"],
    ["decimal64", EVEN, "mul", "123", "1E+5", "1.23E+7"],
    ["decimal128", EVEN, "mul", "0.1", "3", "0.3"],
    ["decimal128", EVEN, "add", "1E+6111", "1E+6111", "2E+6111"],
    ["decimal128", EVEN, "add", "1", "1E-40", "1.000000000000000000000000000000000"],
  ];
  /** The same value: floats by sign too, NaNs alike. */
  const alike = (x: unknown, y: unknown) => Object.is(x, y);
  for (const [format, rounding, operation, a, b, expected] of TABLE) {
    const domain = new D.OfIeee754.Data(format, rounding);
    const result = ev(operation, ...[a, b].filter((v) => v !== null).map((v) => L(v as never, domain)));
    assert(result instanceof D.Value && result.domain.equals(domain) && alike(result.value, expected), `${format} ${a} ${b} ${result}`);
  }
  // The default float keeps the host's arithmetic, which binary64 rounding ties to even is.
  assert(ev("add", 0.1, 0.2) === 0.30000000000000004 && new D.OfIeee754.Data("binary64").equals(D.Float));
}

## DOM-12 · IEEE 754 values compare by totalOrder, cohorts by exponent, and NaNs equal each other

In [ ]:
{
  assert(ev("eq", L("1.5", D64), L("1.5", D64)) === true && ev("eq", L("1.5", D64), L("1.50", D64)) === false);
  assert(ev("lt", L("1.50", D64), L("1.5", D64)) === true && ev("lt", L("-1.5", D64), L("-1.50", D64)) === true);
  assert(ev("lt", L("1.5", D64), L("2", D64)) === true && ev("gt", L("-1", D64), L("-2", D64)) === true);
  assert(ev("lt", L("-0", D64), L("0", D64)) === true && ev("lt", L("-Infinity", D64), L("-9.999999999999999E+384", D64)) === true);
  assert(ev("lt", L("9.999999999999999E+384", D64), L("Infinity", D64)) === true && ev("eq", L("Infinity", D64), L("Infinity", D64)) === true);
  assert(ev("eq", L("NaN", D64), L("NaN", D64)) === true && ev("lt", L("NaN", D64), L("1", D64)) === null);
  assert(ev("eq", L("1", D64), L("NaN", D64)) === false && ev("gt", L("1", D64), L("-1", D64)) === true);
  assert(ev("lt", L("0.1", F128), L("0.2", F128)) === true && ev("eq", L("0.1", F128), L("0.1", F128)) === true);
  assert(ev("le", L(-0, F16), L(0, F16)) === true && ev("eq", L(NaN, F16), L(NaN, F16)) === true);
  // A binary128 value's text is the shortest that rounds back to it.
  assert(is(ev("mul", L("3", F128), L("0.5", F128)), F128, "1.5"));
  assert(is(ev("add", L("0.1", F128), L("0.2", F128)), F128, "0.30000000000000000000000000000000004"));
  assert(is(ev("mul", L("0.5", F128), L("0.5", F128)), F128, "0.25") && is(ev("neg", L("0.49", F128)), F128, "-0.49"));
  assert(is(ev("mul", L("1E+48", F128), L("1E+1", F128)), F128, "1E+49")); // the nearest is below 1E+49
}